# DB061 — Verify Final Dataset Record Counts

**Scope (per Shuhan + Anjum, after flagging the overlap with DB060/DB062):** their reports are *comparison evidence*, not the source of truth. Every count below is computed directly from the real dataset files. Where a report already states a number, it's loaded separately and diffed against our own count — never copied in.

Four counts to produce: **input → enriched → validated → final output**, plus **unique barcodes**, **included**, and **excluded** counts for the v1.1 release specifically (per Anjum).

## Setup
Paths below come from the `git pull` log and known repo structure. Two are marked `CONFIRM` — check these against your actual folder layout before trusting the results.

In [18]:
import json
from pathlib import Path

# Auto-detects the repo root whether this notebook is run from scripts/ or the repo root itself
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "database").exists() and (REPO_ROOT.parent / "database").exists():
    REPO_ROOT = REPO_ROOT.parent
print(f"Repo root resolved to: {REPO_ROOT.resolve()}")

# --- Stage file paths ---
INPUT_PATH = REPO_ROOT / "database" / "seeding" / "products_5k_enriched.json" # CONFIRM: exact folder for the raw source file
ENRICHED_PATH = REPO_ROOT / "database" / "seeding" / "products_enriched.json"  # CONFIRM: vs products_5k_enriched.json — check which is the real enrich-stage output

# --- Validation evidence (v1.1 release) ---
INCLUSION_LEDGER    = REPO_ROOT / "database" / "Release" / "v1.1" / "inclusion_ledger.jsonl"
EXCLUSION_LEDGER    = REPO_ROOT / "database" / "Release" / "v1.1" / "exclusion_ledger.jsonl"
VALIDATION_MANIFEST = REPO_ROOT / "database" / "Release" / "v1.1" / "validation_manifest.json"

# --- Final output ---
FINAL_RELEASE_PATH = REPO_ROOT / "database" / "Release" / "v1.1" / "foodremedy_release_v1.1.json"

# --- Comparison evidence only — never used as the source count ---
DB060_REPORT = REPO_ROOT / "database" / "Reports" / "DB060" / "release_validation.json"
DB062_REPORT = REPO_ROOT / "database" / "Reports" / "DB062" / "pipeline_verification.json"


Repo root resolved to: C:\Users\GAURAV\OneDrive\Desktop\Food Remedy API - Project\Food-Remedy_API


## Helper: inspect a file's real shape before counting
JSON files in this repo sometimes wrap the record list in a key (`products`, `data`, etc.) instead of being a bare array. Run this on any new file first so we count the right thing instead of guessing.

In [19]:
def inspect(path):
    path = Path(path)
    if not path.exists():
        print(f"NOT FOUND: {path}")
        return
    if path.suffix == ".jsonl":
        with open(path, encoding="utf-8") as f:
            lines = [l for l in f if l.strip()]
        print(f"{path.name}: JSONL, {len(lines)} lines")
        if lines:
            print("First record keys:", list(json.loads(lines[0]).keys()))
        return
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):
        print(f"{path.name}: bare JSON array, {len(data)} records")
        if data:
            print("First record keys:", list(data[0].keys()))
    elif isinstance(data, dict):
        print(f"{path.name}: JSON object, top-level keys: {list(data.keys())}")
        for k, v in data.items():
            if isinstance(v, list):
                print(f"  -> '{k}' is a list of {len(v)} items")


Run `inspect()` on every file before trusting a count from it:

In [20]:
for p in [INPUT_PATH, ENRICHED_PATH, INCLUSION_LEDGER, EXCLUSION_LEDGER,
          VALIDATION_MANIFEST, FINAL_RELEASE_PATH, DB060_REPORT, DB062_REPORT]:
    inspect(p)
    print()


products_5k_enriched.json: bare JSON array, 5000 records
First record keys: ['barcode', 'brand', 'productName', 'genericName', 'additives', 'allergens', 'ingredients', 'ingredientsText', 'ingredientsAnalysis', 'categories', 'labels', 'nutrientLevels', 'nutriments', 'nutriscoreGrade', 'productQuantity', 'productQuantityUnit', 'servingQuantity', 'servingQuantityUnit', 'traces', 'tracesFromIngredients', 'completeness', 'images', 'enrichment']

products_enriched.json: bare JSON array, 5000 records
First record keys: ['barcode', 'brand', 'productName', 'genericName', 'additives', 'allergens', 'ingredients', 'ingredientsText', 'ingredientsAnalysis', 'categories', 'labels', 'nutrientLevels', 'nutriments', 'nutriscoreGrade', 'productQuantity', 'productQuantityUnit', 'servingQuantity', 'servingQuantityUnit', 'traces', 'tracesFromIngredients', 'completeness', 'images', 'enrichment', 'allergensDetected', 'dietTags', 'lifestyleTags', 'riskTags', 'moodTags']

inclusion_ledger.jsonl: JSONL, 5000 lin

## Counting functions
Once you've confirmed each file's shape above, pass `record_key` if a file wraps its list (e.g. `record_key="products"`). Leave `None` for a bare array or JSONL.

In [21]:
def count_records(path, record_key=None):
    path = Path(path)
    if path.suffix == ".jsonl":
        with open(path, encoding="utf-8") as f:
            return sum(1 for l in f if l.strip())
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    if record_key:
        return len(data[record_key])
    if isinstance(data, list):
        return len(data)
    raise ValueError(f"{path.name} is a dict with no record_key given — check its keys with inspect() first")


## Stage 1 & 2 — Input and enriched counts

In [22]:
input_count = count_records(INPUT_PATH)
enriched_count = count_records(ENRICHED_PATH)
print(f"Input dataset count:    {input_count}")
print(f"Enriched dataset count: {enriched_count}")


Input dataset count:    5000
Enriched dataset count: 5000


## Stage 3 — Validated count
Taking "validated" as everything that went through validation and got a verdict: included + excluded from the v1.1 ledgers. Cross-check this against whatever total the validation manifest itself states — print its keys first, then pull the real field name once you see it.

In [23]:
included_count = count_records(INCLUSION_LEDGER)
excluded_count = count_records(EXCLUSION_LEDGER)
validated_count = included_count + excluded_count

with open(VALIDATION_MANIFEST, encoding="utf-8") as f:
    manifest = json.load(f)

print(f"Included (ledger):  {included_count}")
print(f"Excluded (ledger):  {excluded_count}")
print(f"Validated (sum):    {validated_count}")
print(f"Manifest top-level keys: {list(manifest.keys())}")
# Once you see the real key names above, compare the manifest's own stated total here by hand


Included (ledger):  5000
Excluded (ledger):  0
Validated (sum):    5000
Manifest top-level keys: ['artifact_sha256', 'configuration', 'database_release_validated', 'dataset', 'delta_summary', 'generated_at_utc', 'generation_base_commit', 'known_limitations', 'production_release_approved', 'production_seeded', 'quality_comparison', 'release_date', 'release_version', 'schema', 'source', 'status', 'ticket', 'validation', 'workflow']


In [24]:
for key in ["dataset", "workflow", "validation", "production_seeded", "production_release_approved", "database_release_validated"]:
    print(f"{key}: {manifest.get(key)}")

dataset: {'file': 'foodremedy_release_v1.1.json', 'records': 5000, 'sha256': '43b955c6837eda262b5b9afba5e8227ae7a1db3538dab110b393b8e9920791e1'}
workflow: {'cleaning': {'conservative_unknown_allergen_records': 1031, 'excluded_records': 0, 'input_records': 5000, 'leading_zero_barcodes_preserved': 256, 'missing_value_policy': "Source nulls and absent nutrient keys are preserved; numeric zero is never substituted for missing evidence. Only a genuinely absent allergen state may be normalised to the conservative ['Unknown'] value.", 'output_records': 5000, 'removed_stale_derived_fields': {}, 'status': 'PASS', 'unique_valid_barcodes': 5000}, 'enrichment': {'failures': 0, 'modules': [{'failures': 0, 'module': 'nutrition_enrich', 'processed': 5000, 'status': 'ok'}, {'failures': 0, 'module': 'allergens', 'processed': 5000, 'status': 'ok'}, {'failures': 0, 'module': 'db009_personalisation_tags', 'processed': 5000, 'status': 'ok'}, {'failures': 0, 'module': 'db021_mood_tags', 'processed': 5000, '

## Stage 4 — Final output count + unique barcodes
Anjum asked specifically for total records, unique barcodes, and included/excluded — counted directly from the real v1.1 release file, not from a report.

In [25]:
with open(FINAL_RELEASE_PATH, encoding="utf-8") as f:
    final_data = json.load(f)

final_records = final_data if isinstance(final_data, list) else final_data.get("products", final_data.get("data"))
final_count = len(final_records)
barcodes = [r.get("barcode") for r in final_records if r.get("barcode")]
unique_barcodes = len(set(barcodes))

print(f"Final output count:   {final_count}")
print(f"Unique barcodes:      {unique_barcodes}")
print(f"Duplicate barcodes:   {final_count - unique_barcodes}")


Final output count:   5000
Unique barcodes:      5000
Duplicate barcodes:   0


## Summary — all stages side by side

In [27]:
summary = {
    "Input": input_count,
    "Enriched": enriched_count,
    "Validated (included+excluded)": validated_count,
    "  - Included": included_count,
    "  - Excluded": excluded_count,
    "Final output": final_count,
    "Unique barcodes (final)": unique_barcodes,
}

for stage, count in summary.items():
    print(f"{stage:<32}{count}")


Input                           5000
Enriched                        5000
Validated (included+excluded)   5000
  - Included                    5000
  - Excluded                    0
Final output                    5000
Unique barcodes (final)         5000


## Cross-check against DB060 / DB062 (comparison evidence, not source)
Load their stated numbers and diff against ours — never overwrite our counts with theirs.

In [28]:
with open(DB060_REPORT, encoding="utf-8") as f:
    db060 = json.load(f)
with open(DB062_REPORT, encoding="utf-8") as f:
    db062 = json.load(f)

print("DB060 report top-level keys:", list(db060.keys()))
print("DB062 report top-level keys:", list(db062.keys()))
# Once you see the real keys, pull their stated counts and diff, e.g.:
# print("DB060 says validated:", db060["total_validated"], "| ours:", validated_count)


DB060 report top-level keys: ['status', 'release_approved', 'total_records', 'valid_records', 'invalid_records', 'count_definition', 'dataset_errors', 'current_validation', 'stored_required_fields', 'quality', 'reason_counts', 'failed_records', 'required_reviews', 'limitations', 'provenance']
DB062 report top-level keys: ['ticket', 'verified_at', 'input', 'output', 'pipeline_config_sha256', 'modules', 'module_failures', 'quality', 'handoff']


In [29]:
from datetime import datetime, timezone
print(datetime.now(timezone.utc).isoformat())

2026-09-29T04:31:13.352507+00:00


In [30]:
import hashlib

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

local_hash = sha256_of(FINAL_RELEASE_PATH)
manifest_hash = manifest["dataset"]["sha256"]

print("Your local file's hash:  ", local_hash)
print("Manifest's stated hash:  ", manifest_hash)
print("Match:", local_hash == manifest_hash)

Your local file's hash:   a1635e222b9e5fbec2ca07fee16bbed1f275a168dc98945744ba1d5f49351949
Manifest's stated hash:   43b955c6837eda262b5b9afba5e8227ae7a1db3538dab110b393b8e9920791e1
Match: False


In [31]:
def sha256_of_normalized(path):
    with open(path, "rb") as f:
        data = f.read()
    data = data.replace(b"\r\n", b"\n")  # undo Windows-style line breaks
    import hashlib
    return hashlib.sha256(data).hexdigest()

normalized_hash = sha256_of_normalized(FINAL_RELEASE_PATH)
print("Normalized hash:", normalized_hash)
print("Matches manifest now:", normalized_hash == manifest["dataset"]["sha256"])

Normalized hash: 43b955c6837eda262b5b9afba5e8227ae7a1db3538dab110b393b8e9920791e1
Matches manifest now: True


## Differences & findings
Document every mismatch here as you find it — this section is your DB061 evidence for the release manifest.

- Input → Enriched: 
- Enriched → Validated: 
- Validated → Final output: 
- vs DB060: 
- vs DB062: 